In [ ]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
from pprint import pprint

# 環境変数の取得 / 1Password管理下にある.envの読み込み
def load_env():
    path = os.path.expanduser("~/.env")

    with open(path, "r", encoding="utf-8") as f:
        env_text = f.read()

    env = {}
    for line in env_text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip()
    return env

# 環境変数の取得
env = load_env()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=env["llm_dev_key"])

# モデル名
MODEL_NAME = "gpt-4o-mini"

# メッセージを格納するリスト
messages=[]

while(True):
    # ユーザーからの質問を受付
    message = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if message.strip()=="":
        break
    print(f"質問:{message}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": message.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # APIへリクエスト
    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        stream=True,
    )

    # 言語モデルからの回答を表示
    response_message = ""
    for chunk in stream:
        if chunk.choices:
            next = chunk.choices[0].delta.content
            if next is not None:
                response_message += next
                print(next, end='', flush=True)

    # メッセージに言語モデルからの回答を追加
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")